# Capillary Rise - benchmark test case (SFB 1194)

Results are published in *A comparative study of transient capillary rise using direct numerical simulations* (https://linkinghub.elsevier.com/retrieve/pii/S0307904X20302134)

In [1]:
#r "D:/rieckmann/BoSSS/BoSSS-Repos/BoSSS-InterfaceSlip/public/src/L4-application/BoSSSpad/bin/Debug/net6.0/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
string proj = "CapillaryRise";
BoSSSshell.WorkflowMgm.Init(proj);
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

Project name is set to 'CapillaryRise'.
Default Execution queue is chosen for the database.
Opening existing database '\\dc3\backup\rieckmann\cluster\databases\CapillaryRise'.


In [3]:
var myDb = BoSSSshell.WorkflowMgm.DefaultDatabase;
var sessions = BoSSSshell.WorkflowMgm.Sessions;

In [4]:
myDb

{ Session Count = 0; Grid Count = 2; Path = \\dc3\backup\rieckmann\cluster\databases\CapillaryRise }

In [5]:
sessions

In [6]:
BoSSSshell.WorkflowMgm.AllJobs

## Test case setup (physical parameters)

In [7]:
bool startUp = true;

In [8]:
string omegaTc = "Omega=1";     // {"Omega=0.1", "Omega=0.5", "Omega=1", "Omega=10", "Omega=100"}

In [9]:
double R = 5.0e-3;      // in cm
double H = 4.0e-2;

double rhoA = 0.0;
double muA = 0.01;
double rhoB = 0.0;
double muB = 0.01 / 1000;
double sigma = 0.0;

double Lslip = R / 5.0;
double betaS_A = (muA / Lslip);
double betaS_B = (muB / Lslip);

double betaL = 0;
double theta_e = 3.0 * Math.PI / 18.0;

double g = 0.0;
Formula GravityValue;

double t_end = 0.0;
double t_startUp = 0.0;

double dt = 0.0;
double dt_startUp = 0.0;

Guid restartID = new Guid();
int ts_restart = 0;

In [10]:
switch(omegaTc) {
    case "Omega=0.1": {

            rhoA = 1663.8;
            rhoB = 1663.8 / 1000;      
            sigma = 0.2;       // kg / s^2

            g = 1.04;         // cm / s^2
            GravityValue = new Formula(
                "GravY",
                false,
                "double GravY(double[] X) { return -1.04; } "
            );

            t_end = 13.86;
            t_startUp = 0.0; 

            dt = 2e-5; 
            dt_startUp = 3e-5;

            //restartID = new Guid("");   // startUp session

            break;
        }
    case "Omega=0.5": {

            rhoA = 133.0;
            rhoB = 133.0 / 1000;      
            sigma = 0.1;       // kg / s^2

            g = 6.51;         // cm / s^2
            GravityValue = new Formula(
                "GravY",
                false,
                "double GravY(double[] X) { return -6.51; } "
            );

            t_end = 1.11;
            t_startUp = 0.0;

            dt = 2e-5;
            dt_startUp = 4e-5;

            //restartID = new Guid("");   // startUp session

            break;
        }
    case "Omega=1": {

            rhoA = 83.1;
            rhoB = 83.1 / 1000;      
            sigma = 0.04;       // kg / s^2

            g = 4.17;         // cm / s^2
            GravityValue = new Formula(
                "GravY",
                false,
                "double GravY(double[] X) { return -4.17; } "
            );

            t_end = 0.7;
            t_startUp = 0.0;

            dt = 1e-5;
            dt_startUp = 5e-4;

            //restartID = new Guid("");   // startUp session

            break;
        }
    case "Omega=10": {

            rhoA = 3.3255;
            rhoB = 3.3255 / 1000;      
            sigma = 0.01;       // kg / s^2

            g = 26.042;         // cm / s^2
            GravityValue = new Formula(
                "GravY",
                false,
                "double GravY(double[] X) { return -26.042; } "
            );

            t_end = 2.7713;
            t_startUp = 0.0;

            dt = 4e-4;
            dt_startUp = 1e-5;

            //restartID = new Guid("");   // startUp session

            break;
        }
    case "Omega=100": {

            rhoA = 0.33255;
            rhoB = 0.33255 / 1000;      
            sigma = 0.001;       // kg / s^2

            g = 26.042;         // cm / s^2
            GravityValue = new Formula(
                "GravY",
                false,
                "double GravY(double[] X) { return -26.042; } "
            );

            t_end = 27.713;
            t_startUp = 0.0;

            dt = 1e-3;
            dt_startUp = 1e-3;

            //restartID = new Guid("");   // startUp session

            break;
        }
} 

## Grid creation

In [11]:
int[] Resolutions = new int[] { 4 };    // cells per radius R { 1, 2, 4, 8, 16 };
IGridInfo[] Grids = new IGridInfo[Resolutions.Length];

bool useSymmetry = true;

for(int i = 0; i < Resolutions.Length; i++) {
    int Res = Resolutions[i];

    string GridName;
    if(useSymmetry)
        GridName = $"CapillaryRise_{Res}x{8*Res}_halfTube";
    else
        GridName = $"CapillaryRise_{Res}x{8*Res}_fullTube";

    //IGridInfo cachedGrid = wmg.Grids.FirstOrDefault(grid => grid.Name == GridName);
    IGridInfo cachedGrid = null;
    if(cachedGrid == null) {
        
        // must create new Grid
        double[] Xnodes;
        if(useSymmetry)
            Xnodes = GenericBlas.Linspace(0, R, Res + 1);
        else
            Xnodes = GenericBlas.Linspace(-R, R, (2 * Res) + 1);
        double[] Ynodes = GenericBlas.Linspace(0, H, 8 * Res + 1);
        var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes);
    
        grd.EdgeTagNames.Add(1, "wall_lower");
        grd.EdgeTagNames.Add(2, "pressure_outlet_upper");

        if(useSymmetry)
            grd.EdgeTagNames.Add(3, "slipsymmetry_left");
        else
            grd.EdgeTagNames.Add(3, "navierslip_linear_left");

        grd.EdgeTagNames.Add(4, "navierslip_linear_right");

        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 0;
            if(Math.Abs(X[1]) <= 1.0e-8)
                et = 1;
            if(Math.Abs(X[1] - H) <= 1.0e-8)
                et = 2;
            if(useSymmetry) {
                if(Math.Abs(X[0]) <= 1.0e-8)
                    et = 3;
            } else {
                if(Math.Abs(X[0] + R) <= 1.0e-8)
                    et = 3;
            }
            if(Math.Abs(X[0] - R) <= 1.0e-8)
                et = 4;

            return et;
        });     
        
        grd.Name = GridName;
        
        BoSSSshell.WorkflowMgm.DefaultDatabase.SaveGrid(ref grd);
        Grids[i] = grd;
        
    } else {
        //Console.WriteLine($"type: {cachedGrid.GetType()}, is IGridInfo? {cachedGrid is IGridInfo}");
        Console.WriteLine("Grid already found in database - identifid by name " + GridName);
        Grids[i] = cachedGrid;
    }
}

Grid Edge Tags changed.
An equivalent grid (a77d09c1-7248-4c80-9532-10e5f2435ad7) is already present in the database -- the grid will not be saved.


## Control settings

In [12]:
List<XNSE_Control> Controls = new List<XNSE_Control>();
Controls.Clear();

In [13]:
for(int gIdx = 0; gIdx < Resolutions.Length; gIdx++) { 

var C = new XNSE_Control();

int k = 2;
C.SetDGdegree(k);
C.savetodb = true;

// physical parameters
// ===================
C.PhysicalParameters.rho_A = rhoA;
C.PhysicalParameters.mu_A = muA;
C.PhysicalParameters.rho_B = rhoB;
C.PhysicalParameters.mu_B = muB;

C.PhysicalParameters.Sigma = sigma;

C.PhysicalParameters.betaS_A = betaS_A;
C.PhysicalParameters.betaS_B = betaS_B;

C.PhysicalParameters.betaL = betaL;
C.PhysicalParameters.theta_e = theta_e;

C.PhysicalParameters.IncludeConvection = !startUp;
C.PhysicalParameters.Material = true;

    
// startUp?
// ========
if (startUp) {
    // set grid
    C.SetGrid(Grids[gIdx]);
    
    // initial conditions
    double h0 = 1e-2;
    //Func<double[], double> PhiFunc = (X => X[1] - h0);
    Formula PhiFunc = new Formula(
        "Phi",
        false,
        "double Phi(double[] X) { return X[1] - 1e-2; } "
    );
    C.AddInitialValue("Phi", PhiFunc);

    //Func<double[], double> GravFunc = (X => -g);
    C.AddInitialValue("GravityY#A", GravityValue);
    C.AddInitialValue("GravityY#B", GravityValue);

} else {

    if(ts_restart > 0)
        C.RestartInfo = new Tuple<Guid, BoSSS.Foundation.IO.TimestepNumber>(restartID, ts_restart);
    else
        C.RestartInfo = new Tuple<Guid, BoSSS.Foundation.IO.TimestepNumber>(restartID, null);

}

// boundary conditions
// ===================
if(startUp) {
    C.AddBoundaryValue("wall_lower");
} else {
    C.AddBoundaryValue("wall_lower");
    C.ChangeBoundaryCondition("wall_lower", "pressure_outlet_lower");
    //C.AddBoundaryValue("pressure_outlet_lower");
}
C.AddBoundaryValue("pressure_outlet_upper");

if(useSymmetry)
    C.AddBoundaryValue("slipsymmetry_left");
else
    C.AddBoundaryValue("navierslip_linear_left");

C.AddBoundaryValue("navierslip_linear_right");

C.AdvancedDiscretizationOptions.GNBC_Localization = NavierSlip_Localization.Bulk;
C.AdvancedDiscretizationOptions.GNBC_SlipLength = NavierSlip_SlipLength.Prescribed_SlipLength;
C.PhysicalParameters.sliplength = Lslip;


// misc. solver options
// ====================
C.LSContiProjectionMethod = ContinuityProjectionOption.ConstrainedDG;

C.NonLinearSolver.SolverCode = NonLinearSolverCode.Picard;
C.NonLinearSolver.ConvergenceCriterion = 1e-8;
C.NonLinearSolver.MaxSolverIterations = 20;
C.LevelSet_ConvergenceCriterion = 1e-6;

C.AdvancedDiscretizationOptions.FilterConfiguration = CurvatureAlgorithms.FilterConfiguration.Default;
C.AdvancedDiscretizationOptions.SurfStressTensor = SurfaceSressTensor.Isotropic;
C.AdvancedDiscretizationOptions.SST_isotropicMode = SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;

{
    C.AdaptiveMeshRefinement = true;
    int AMRlevel = 1;
    C.activeAMRlevelIndicators.Add(new AMRonNarrowband() { maxRefinementLevel = AMRlevel });
    C.activeAMRlevelIndicators.Add(new AMRLevelIndicatorLibrary.AMRonBoundary(new byte[] { 4 }) { maxRefinementLevel = AMRlevel });
    if(!useSymmetry)
        C.activeAMRlevelIndicators.Add(new AMRLevelIndicatorLibrary.AMRonBoundary(new byte[] { 3 }) { maxRefinementLevel = AMRlevel });
    C.AMR_startUpSweeps = AMRlevel;
}


// level-set
// =========
C.Option_LevelSetEvolution = LevelSetEvolution.FastMarching;


// Timestepping
// ============
C.TimeSteppingScheme = TimeSteppingScheme.ImplicitEuler;
C.Timestepper_LevelSetHandling = LevelSetHandling.Coupled_Once;

C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
C.dtFixed = (startUp) ? dt_startUp : dt;
C.Endtime = (startUp) ? Math.Max(Math.Sqrt(2 * R / g), 2 * R * muA / sigma) * 4.0 : (t_startUp + t_end);
C.NoOfTimesteps = (int)(C.Endtime / C.dtMin);
C.saveperiod = 100;

        
    
C.PostprocessingModules.Add(new BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases.MovingContactLineLogging() { LogPeriod = 100 });
    
double Res = Resolutions[gIdx];
if (startUp) 
    C.SessionName = $"CapillaryRise_{omegaTc}_{Res}x{8*Res}AMR1_startUp";
else
    C.SessionName = $"CapillaryRise_{omegaTc}_{Res}x{8*Res}AMR1";
    
    
Controls.Add(C);

}


(35,12): warning CS0219: The variable 'h0' is assigned but its value is never used



## Launch job

In [14]:
Controls.Select(C => C.SessionName)

index,value
0,CapillaryRise_Omega=1_4x32AMR1_startUp


In [15]:
var job = Controls.First().CreateJob();

In [18]:
//wmg.ResetProject(true, true, true, true);

Deleting Sessions in projects...
Deleting Grids in projects...
Opening existing database '\\hpccluster\hpccluster-scratch\rieckmann\CapillaryRise'.
Opening existing database '\\dc3\userspace\rieckmann\cluster\CapillaryRise'.
Opening existing database '\\hpccluster\hpccluster-scratch\rieckmann\BoSSS_DB\CapillaryRise'.
Opening existing database 'D:\rieckmann\BoSSS\MiniBatchProcessor\databases\CapillaryRise'.
Opening existing database '\\FDYGITRUNNER\ValidationTests\databases\CapillaryRise'.
Opening existing database 'D:\rieckmann\BoSSS\DeploymentTest\CapillaryRise'.
Grid a77d09c1-7248-4c80-9532-10e5f2435ad7 deleted.
Grid 027b2a9e-7943-485e-bd8f-b120f8e40fc2 deleted.
Deleting Job deployments in projects...
Forgetting all Jobs defined in this notebook so far...


In [17]:
foreach(var ctrl in Controls) {
        //var job              = ctrl.CreateJob();
     job.NumberOfMPIProcs = 1;
     job.Activate();
}

unable to determine job status - unknown
unable to determine job status - unknown


Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job CapillaryRise_Omega=1_4x32AMR1_startUp ... 
Deploying executables and additional files ...


unable to determine job status - unknown


Deployment directory: \\dc3\backup\rieckmann\cluster\binaries\CapillaryRise-XNSE_Solver2024Feb09_120650.680536


unable to determine job status - unknown


copied 43 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.

